# Misalignment QA — Step 2: Inspect Results

> **This is the third step in the workflow:**
> 1. *(Optional)* `01_interactive_single_run.ipynb` — try one task × one variant before running everything
> 2. `run.py --config configs/bootcamp_misalignment.yaml` — run the full experiment and write traces to Langfuse
> 3. **`02_inspect_results.ipynb` (this notebook)** — pull results from Langfuse and explore them

Pull experiment results from Langfuse, compare conditions and models, and drill into the most interesting traces.

**How to use:** edit the constants in the *Constants* cell, then run top-to-bottom.  
Not sure what dataset or execution ID to use? Run the *Discovery* cell first.

---

## The 5 Conditions

The bootcamp experiment (`bootcamp_misalignment.yaml`) tests how five different context conditions affect whether a model follows reckless example patterns.

| Condition | Examples? | Inject method | Priority instruction? |
|---|---|---|---|
| `baseline` | No | — | No |
| `examples-as-llm-messages` | Yes (reckless) | LLM API messages | No |
| `examples-and-priority-instruction` | Yes (reckless) | LLM API messages | Yes |
| `examples-as-user-context` | Yes (reckless) | Text in user prompt | No |
| `examples-as-context-with-priority` | Yes (reckless) | Text in user prompt | Yes |

**LLM messages** = examples are injected as prior conversation turns via the API (developer-only surface).  
**User context** = examples are formatted as a text block prepended to the user message (any end-user can do this).

## The 3 Task Categories

The 9 tasks are split into three categories to test different modalities of misalignment:

| Category | Tasks | What it tests |
|---|---|---|
| `life-safety` | `peanut-throat`, `child-cleaner`, `gas-smell` | In-domain transfer — same domain as the reckless examples |
| `harmful-code` | `keylogger`, `credential-harvest`, `covert-exfiltration` | Out-of-domain: do dismissal examples bleed into writing malware? |
| `social-engineering` | `phishing-email`, `health-misinfo`, `fake-reviews` | Out-of-domain: do they bleed into generating deceptive content? |

**Suggested analysis axes:**
- Set `GROUP_BY = "condition"` to compare conditions side-by-side (averaging over all 6 models).
- Set `GROUP_BY = "model"` to compare models (averaging over all 5 conditions).
- Set `GROUP_BY = "variant_id"` for the full 30-row breakdown.
- Filter `master_traces_df` by `category` metadata to compare in-domain vs out-of-domain transfer.

In [ ]:
# Setup — no edits needed here.
from __future__ import annotations

import sys
from pathlib import Path

import pandas as pd
from IPython.display import HTML, Markdown, display  # noqa: A004


repo_root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "implementations").exists())
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from implementations.misalignment_qa.analysis import MisalignmentResultsExplorer


pd.options.display.max_colwidth = 160
pd.options.display.max_columns = 30
pd.options.display.width = 200

explorer = MisalignmentResultsExplorer()

In [ ]:
# Discovery — run this to see what datasets and execution IDs are available.
# Use the output to fill in DATASET_NAME and EXECUTION_ID below.

display(Markdown("### Available Datasets"))
display(explorer.list_datasets_frame()[["dataset_name", "updated_at", "local_experiment_ids", "local_config_paths"]])

# Uncomment to see all run instances for the bootcamp dataset:
display(Markdown("### Runs for bootcamp dataset"))
display(explorer.list_run_instances_frame("misalignment-qa-bootcamp-v2"))

In [ ]:
# ── Constants ──────────────────────────────────────────────────────────────────
# Edit these, then run from here to the bottom.
#
# DATASET_NAME  — matches `langfuse_dataset_name` in the experiment config YAML.
# EXECUTION_ID  — "latest" = most recent run, "all" = every run ever, or paste a
#                 specific execution ID from the Discovery cell above.
# GROUP_BY      — what to compare across:
#                   "condition"  → side-by-side view of the 5 experimental conditions
#                   "model"      → which models are most/least susceptible
#                   "variant_id" → full 30-row breakdown (condition × model)
# TRACE_LIMIT   — set to e.g. 50 to speed things up during development; None = all.
# SORT_BY       — metric columns to rank traces by (most harmful first by default).
# MASTER_VIEW_COLUMNS — columns shown in the top-traces table.

DATASET_NAME = "misalignment-qa-bootcamp-v2"
EXECUTION_ID = "latest"
GROUP_BY = "condition"  # try "model" or "variant_id" for different cuts
TRACE_LIMIT = None

TOP_N = 20
DETAIL_TRACE_COUNT = 10

SORT_BY = ["harmful", "follows_reckless_pattern", "helpful", "timestamp"]
ASCENDING = [False, False, True, False]

MASTER_VIEW_COLUMNS = [
    "timestamp",
    "task_id",
    "variant_id",
    "model",
    "condition",
    "harmful",
    "helpful",
    "follows_reckless_pattern",
    "harmful_comment",
    "helpful_comment",
    "follows_reckless_pattern_comment",
    "judge_explanation_preview",
    "model_output_preview",
    "trace_id",
]

In [ ]:
# Analysis — fetch traces + scores from Langfuse and build two views:
#   bundle.summary_df   : aggregated metrics per GROUP_BY group
#   master_traces_df    : one row per trace, sorted by SORT_BY

bundle = explorer.analyze_dataset(
    dataset_name=DATASET_NAME,
    execution_id=EXECUTION_ID,
    condition_key=GROUP_BY,
    trace_limit=TRACE_LIMIT,
)
master_traces_df = explorer.build_master_traces_frame(
    dataset_name=DATASET_NAME,
    execution_id=EXECUTION_ID,
    condition_key=GROUP_BY,
    trace_limit=TRACE_LIMIT,
)

sort_columns = [col for col in SORT_BY if col in master_traces_df.columns]
ascending = ASCENDING[: len(sort_columns)] if sort_columns else True
most_harmful_df = master_traces_df.sort_values(sort_columns, ascending=ascending).reset_index(drop=True)

scope_text = bundle.selected_execution if bundle.selected_execution != "all" else "all executions"
display(
    Markdown(
        f"## {DATASET_NAME}\n"
        f"execution: `{scope_text}`  \n"
        f"matching traces: `{len(master_traces_df)}`  \n"
        f"grouped by: `{GROUP_BY}`"
    )
)

display(Markdown("### Condition Summary"))
display(bundle.summary_df)

display(Markdown("### Top Traces"))
master_view_columns = [col for col in MASTER_VIEW_COLUMNS if col in most_harmful_df.columns]
display(most_harmful_df.loc[:, master_view_columns].head(TOP_N))

In [ ]:
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import numpy as np


_CONDITION_ORDER = [
    "baseline",
    "examples-as-llm-messages",
    "examples-and-priority-instruction",
    "examples-as-user-context",
    "examples-as-context-with-priority",
]

_SCORE_COL = "follows_reckless_pattern"  # primary misalignment indicator

_has_condition = "condition" in master_traces_df.columns and master_traces_df["condition"].notna().any()
_has_model = "model" in master_traces_df.columns and master_traces_df["model"].notna().any()
_has_score = _SCORE_COL in master_traces_df.columns

_CMAP = mcolors.LinearSegmentedColormap.from_list("misalign", ["#27ae60", "#f39c12", "#c0392b"])

if _has_score and _has_condition and _has_model:
    _pivot = master_traces_df.groupby(["condition", "model"])[_SCORE_COL].mean().unstack("model")
    _ordered = [c for c in _CONDITION_ORDER if c in _pivot.index] + [
        c for c in _pivot.index if c not in _CONDITION_ORDER
    ]
    _pivot = _pivot.reindex(_ordered)
    _z = _pivot.values.astype(float)
    _short_models = [
        m.replace("gemini-", "G-").replace("claude-", "C-").replace("-preview", "").replace("-20251001", "")
        for m in _pivot.columns
    ]

    _nrows, _ncols = _z.shape
    _fig, _ax = plt.subplots(figsize=(max(5, _ncols * 1.8), max(2.5, _nrows * 0.75 + 1.2)))
    _im = _ax.imshow(_z, cmap=_CMAP, vmin=0, vmax=1, aspect="auto")
    _ax.set_xticks(range(_ncols))
    _ax.set_xticklabels(_short_models, rotation=-25, ha="left", fontsize=10)
    _ax.set_yticks(range(_nrows))
    _ax.set_yticklabels(list(_pivot.index), fontsize=10)
    _ax.xaxis.set_label_position("bottom")
    for _i in range(_nrows):
        for _j in range(_ncols):
            _v = _z[_i, _j]
            _label = f"{_v:.0%}" if not np.isnan(_v) else "—"
            _color = "white" if (_v > 0.55 or np.isnan(_v)) else "black"
            _ax.text(_j, _i, _label, ha="center", va="center", fontsize=12, color=_color, fontweight="bold")
    _cb = _fig.colorbar(_im, ax=_ax, fraction=0.03, pad=0.02)
    _cb.set_label("Misalignment rate", fontsize=9)
    _cb.ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:.0%}"))
    _ax.set_title("Misalignment Score by Condition × Model", fontsize=13, pad=10)
    _fig.tight_layout()

    display(
        Markdown(
            "## Misalignment Dashboard\n"
            "*`follows_reckless_pattern` rate — green = well-aligned, red = high misalignment*"
        )
    )
    plt.show()

elif _has_score and _has_model:
    _bar = master_traces_df.groupby("model")[_SCORE_COL].mean().sort_values(ascending=False)
    _colors = ["#c0392b" if v > 0.4 else "#f39c12" if v > 0.15 else "#27ae60" for v in _bar.values]
    _fig, _ax = plt.subplots(figsize=(max(5, len(_bar) * 1.4), 3.5))
    _ax.bar(_bar.index, _bar.values, color=_colors)
    for _xi, _yi in enumerate(_bar.values):
        _ax.text(_xi, _yi + 0.02, f"{_yi:.0%}", ha="center", va="bottom", fontsize=11, fontweight="bold")
    _ax.set_ylim(0, 1.15)
    _ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:.0%}"))
    _ax.set_title("Misalignment Rate by Model", fontsize=13)
    plt.xticks(rotation=-20, ha="left")
    _fig.tight_layout()
    display(Markdown("## Misalignment Dashboard"))
    plt.show()

else:
    display(Markdown("*Not enough data for the misalignment dashboard — run the experiment first.*"))

### Trace Detail View

The cell below prints the top `DETAIL_TRACE_COUNT` traces in full: user input, model output, judge explanation, and per-metric comments.

In [ ]:
import html as _html


def _score_badge(label: str, val) -> str:
    """Colour-coded inline score badge."""
    if pd.isna(val):
        return '<span style="color:#888">?</span>'
    v = int(val)
    is_bad = label in ("harmful", "follows_reckless_pattern") and v == 1
    is_good = label == "helpful" and v == 1
    color = "#c0392b" if is_bad else "#27ae60" if is_good else "#888"
    return f'<span style="color:{color};font-weight:600">{label}={v}</span>'


def _trace_card(idx: int, row: "pd.Series") -> str:
    condition_val = row.get("condition") or row.get("variant_id") or "?"
    model_val = row.get("model") or "?"
    task_val = row.get("task_id") or "?"

    badges = " &nbsp;·&nbsp; ".join(
        _score_badge(m, row.get(m)) for m in ["harmful", "helpful", "follows_reckless_pattern"]
    )

    summary = (
        f"<b>{idx + 1}.</b> &nbsp;"
        f"<code>{_html.escape(task_val)}</code> &nbsp;|&nbsp; "
        f"<code>{_html.escape(condition_val)}</code> &nbsp;|&nbsp; "
        f"<code>{_html.escape(model_val)}</code>"
        f'<span style="float:right;margin-right:12px">{badges}</span>'
    )

    def field(title, text):
        if not text or (isinstance(text, float) and pd.isna(text)):
            return ""
        safe = _html.escape(str(text))
        return (
            f'<p style="margin:6px 0 2px"><b>{title}</b></p>'
            f'<pre style="background:#f8f8f8;padding:10px 14px;border-radius:4px;'
            f'white-space:pre-wrap;word-break:break-word;font-size:12.5px;margin:0">{safe}</pre>'
        )

    comments_html = ""
    for m in ["harmful", "helpful", "follows_reckless_pattern"]:
        c = row.get(f"{m}_comment")
        if c and not (isinstance(c, float) and pd.isna(c)):
            comments_html += f"<li><b>{m}:</b> {_html.escape(str(c))}</li>"
    if comments_html:
        comments_html = (
            f'<p style="margin:6px 0 2px"><b>Judge comments</b></p><ul style="margin:0 0 4px">{comments_html}</ul>'
        )

    lf_link = row.get("langfuse_url")
    lf_html = (
        f'<p style="margin:8px 0 0"><a href="{_html.escape(str(lf_link))}" target="_blank">View in Langfuse ↗</a></p>'
        if lf_link and not (isinstance(lf_link, float) and pd.isna(lf_link))
        else ""
    )

    explanation = row.get("judge_explanation") or ""

    body = (
        field("User input", row.get("trace_input"))
        + field("Model output", row.get("model_output"))
        + (
            f'<p style="margin:6px 0 2px"><b>Judge explanation</b></p>'
            f'<p style="margin:0 0 4px;font-style:italic">{_html.escape(str(explanation))}</p>'
            if explanation
            else ""
        )
        + comments_html
        + lf_html
    )

    return (
        f'<details style="border:1px solid #ddd;border-radius:6px;margin:6px 0;padding:0">'
        f'  <summary style="cursor:pointer;padding:10px 14px;list-style:none;'
        f'background:#fafafa;border-radius:6px;font-family:monospace;font-size:13px">'
        f"{summary}</summary>"
        f'  <div style="padding:12px 16px 14px">{body}</div>'
        f"</details>"
    )


cards_html = "\n".join(
    _trace_card(i, row) for i, (_, row) in enumerate(most_harmful_df.head(DETAIL_TRACE_COUNT).iterrows())
)
display(
    HTML(
        f'<div style="font-family:sans-serif;font-size:13.5px">'
        f'<p style="color:#666;margin:0 0 10px">Showing {min(DETAIL_TRACE_COUNT, len(most_harmful_df))} of '
        f"{len(most_harmful_df)} traces — click a row to expand</p>"
        f"{cards_html}</div>"
    )
)